In [0]:
%run ../../config/utils

In [0]:
import argparse
import csv
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
import yaml

import lib_trip_spend.python_general_utilities as util_func
import pandas as pd
import mlflow
from pyspark.sql import Row
from pyspark.sql import functions as f
mlflow.autolog(disable=True)

In [0]:
############## SET VARIABLES FROM CONFIG ########################################
with open('./config/config.yml', "r") as stream:
    config = yaml.load(stream, Loader=yaml.FullLoader)

model = None if not dbutils.widgets.get("model") else dbutils.widgets.get("model")

START_WINDOW = config["trip"]["start_window"] - 1
END_WINDOW = config["trip"]["end_window"] - 1
SPLIT_RATE = config["shared"]["split_rate"]
SAMPLE_RATE = config["shared"]["sample_rate"]
NUMBER_OF_TREES = config["grid_search"]["number_of_trees"]
MAX_DEPTHS = config["grid_search"]["max_depths"]
MAX_FEATURES = config["grid_search"]["max_features"]
MIN_LEAF_SIZES = config["grid_search"]["min_leaf_size"]
MODEL = config["grid_search"]["model"]
DATE = datetime.today().date()

if model is not None:
    MODEL = model

In [0]:
############## RUN SCRIPT #############################################
if MODEL == "cont":
    label_column = "spend_from_%s_%s" % (str(START_WINDOW), str(END_WINDOW))
if MODEL == "bin":
    label_column = "will_visit_from_%s_%s" % (
        str(START_WINDOW),
        str(END_WINDOW),
    )
    category_cube = pd.read_csv(feature_path)
dependent_columns = ["will_visit_from_5_7", "spend_from_5_7"]
features = pd.read_csv(feature_path)
(
    column_headers,
    categorical_features,
    continious_features,
) = util_func.get_column_headers(features, dependent_columns)
print("---------------------1/6 read in transformed file---------------------")
data = spark.table(model_trip_spend_etl_output).select(*column_headers).toPandas()
data = data.fillna(0)
print(
    "---------------------2/6 split data in test and train---------------------"
)

training, testing = util_func.test_and_train_to_pandas(
    data, SPLIT_RATE, SAMPLE_RATE
)

model_metrics_names = (
    config["trip"]["metrics"]
    if MODEL == "bin"
    else config["spend"]["metrics"]
)

training, testing = util_func.train_initial_model(
    training.drop(columns=dependent_columns),
    training[["will_visit_from_5_7"]],
    testing.drop(columns=dependent_columns),
    SPLIT_RATE,
    testing[["will_visit_from_5_7"]],
    label_column,
)

In [0]:
experiment_name = experiment_name_trip_spend_grid_search

mlflow.sklearn.autolog(disable=False, log_input_examples=True, log_models=True, log_datasets=False)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
run = mlflow.start_run(run_name=f'training_{MODEL}_{mark_datetime}')

feature_dataset = mlflow.data.from_spark(spark.table(model_trip_spend_etl_output).select(*column_headers), name = 'trips_spend_etl_dataset')

In [0]:
for max_depth in MAX_DEPTHS:
    for max_feature in MAX_FEATURES:
        for numer_estimators in NUMBER_OF_TREES:
            for min_sample in MIN_LEAF_SIZES:
                payload = {
                    "mlflow_run_id": run.info.run_id,
                    "model": MODEL,
                    "date": DATE,
                }
                with mlflow.start_run(nested=True) as run_nested:
                    payload['mlflow_nested_run_id'] = run_nested.info.run_id
                    mlflow.log_input(feature_dataset, context="source")
                    mlflow.log_input(mlflow.data.from_pandas(training, source=feature_dataset.source), context="training")
                    mlflow.log_input(mlflow.data.from_pandas(testing, source=feature_dataset.source), context="testing")
                    print(max_depth, max_feature, numer_estimators, min_sample)
                    if MODEL == "cont":
                        model = RandomForestRegressor(
                            n_jobs=-1,
                            max_depth=max_depth,
                            max_features=max_feature,
                            warm_start=True,
                            n_estimators=numer_estimators,
                            min_samples_leaf=min_sample,
                        )
                        model.fit(
                            training.drop(columns=[label_column]),
                            training[[label_column]].values.ravel(),
                        )
                        predictions = model.predict(
                            testing.drop(columns=[label_column])
                        )
                        model_metrics = (
                            util_func.create_spend_propensity_metric(
                                predictions, testing[[label_column]]
                            )
                        )
                    else:
                        model = RandomForestClassifier(
                            n_jobs=-1,
                            max_depth=max_depth,
                            max_features=max_feature,
                            warm_start=True,
                            n_estimators=numer_estimators,
                            min_samples_leaf=min_sample,
                        )

                        model.fit(
                            training.drop(columns=[label_column]),
                            training[[label_column]].values.ravel(),
                        )
                        predictions = model.predict(
                            testing.drop(columns=[label_column])
                        )
                        model_metrics = (
                            util_func.create_trip_propensity_metric(
                                predictions, testing[[label_column]]
                            )
                        )
                    payload["max_depth"] = int(max_depth)
                    payload["max_feature"] = int(max_feature)
                    payload["number_estimators"] = int(numer_estimators)
                    payload["min_leaf"] = int(min_sample)
                    for k, v in zip(model_metrics_names, model_metrics): payload[k] = float(v)
                    
                    df_row = spark.createDataFrame([Row(**payload)])
                    df_row.write.mode("append").option("mergeSchema", "true").saveAsTable(trip_spend_grid_search)

In [0]:
mlflow.end_run()